In [65]:
'''
NOTE: 4/23/2026 suspect that there is an issue with this function causing unreliable downstream results. 
Rebuilding the function from scratch to investigate.
'''
# def combine_uh_doan_stops(df, max_gap_minutes = 5, combined_stop_id = 999):
#     # combine UH and Doan stops for capacity
#     # Carmack -> UH is inbound, Doan -> Carmack is outbound
#     # Treating UH and Doan as one stop.
#     """
#     Combine University Hospital and Doan stops into a single stop in the stop inventory for metrics.

#     Args:
#         df (DataFrame): The consolidated bus state DataFrame containing stop-level data.
#         max_gap_minutes (int, optional): The maximum time gap in minutes between the UH and Doan stops to consider them as a pair. Defaults to 5 minutes.

#     Returns:
#         DataFrame: A DataFrame with University Hospital and Doan stops combined into a single stop with ID of 999.
#     """

'\nNOTE: 4/23/2026 suspect that there is an issue with this function causing unreliable downstream results. \nRebuilding the function from scratch to investigate.\n'

In [66]:
# read in df passed to function
import pandas as pd
import numpy as np
import pathlib

df = pd.read_csv("K:/AP/TTM/Data/WMC_Dashboard/mc_busstate_consolidated_MAR_2026.csv")

In [67]:
# df is the consolidated busstate data for the medical center route
"""
Combine University Hospital and Doan stops into a single stop in the stop inventory for metrics.

Args:
    df (DataFrame): The consolidated bus state DataFrame containing stop-level data.
    max_gap_minutes (int, optional): The maximum time gap in minutes between the UH and Doan stops to consider them as a pair. Defaults to 5 minutes.

Returns:
    DataFrame: A DataFrame with University Hospital and Doan stops combined into a single stop with ID of 999.
"""
max_gap_minutes = 5
combined_stop_id = 999

df.describe()

,BUS_ID,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,HOUR,MINUTE
count,36110.000000,36110.000000,36110.000000,36110.000000,36110.000000,36110.000000,36110.000000,36110.000000,36110.000000
mean,2053.524924,18038.376156,247.019884,3.401080,3.338909,10.731681,1504.345749,12.686320,29.153586
std,392.258042,10415.936563,165.675538,5.499187,5.242600,10.677985,2.897342,6.177908,17.491762
min,1303.000000,1.000000,37.000000,0.000000,0.000000,0.000000,1501.000000,0.000000,0.000000
25%,1802.000000,9018.250000,94.000000,0.000000,0.000000,3.000000,1502.000000,7.000000,14.000000
50%,2003.000000,18034.500000,401.000000,1.000000,1.000000,7.000000,1504.000000,14.000000,29.000000
75%,2501.000000,27057.750000,403.000000,5.000000,4.000000,16.000000,1506.000000,18.000000,44.000000
max,2503.000000,36083.000000,404.000000,51.000000,44.000000,74.000000,1514.000000,23.000000,59.000000


In [54]:
df['STOP_ID'].value_counts()
# goal: combine ajdacent UH and Doan stops into one artificial stop
# can do this by:
# 1. Identify pairs of UH and Doan stops that occur within a certain time window (e.g., 5 minutes) of each other.
# 2. For each identified pair, create a new stop entry with a unique stop ID (e.g., 999) and assign the relevant attributes (e.g., timestamp, bus ID) from the original stops.
# 3. Remove the original UH and Doan stop entries from the DataFrame and replace them with the new combined stop entry.

STOP_ID
404    6417
403    6411
37     6363
401    6230
95     5360
94     5329
Name: count, dtype: int64

In [55]:
# innvestigate a single run to see how the stops look in sequence
df[(df['RUN_ID'] == 1501) & (df['DATE'] == '2026-03-02')].sort_values(by = 'ARRIVAL').tail(20)

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE
14723,1904,2026-03-02,14707,403,0.0,2.0,9,1501.0,MC,1900-01-01 22:56:56,1900-01-01 22:57:44,0 days 00:00:48,22,56
14724,1904,2026-03-02,14708,404,0.0,0.0,9,1501.0,MC,1900-01-01 22:58:34,1900-01-01 22:59:07,0 days 00:00:33,22,58
14725,1904,2026-03-02,14709,94,0.0,2.0,7,1501.0,MC,1900-01-01 23:00:01,1900-01-01 23:03:13,0 days 00:03:12,23,0
14726,1904,2026-03-02,14710,95,0.0,0.0,7,1501.0,MC,1900-01-01 23:03:49,1900-01-01 23:04:16,0 days 00:00:27,23,3
14727,1904,2026-03-02,14711,401,0.0,0.0,7,1501.0,MC,1900-01-01 23:09:00,1900-01-01 23:09:09,0 days 00:00:09,23,9
14728,1904,2026-03-02,14712,37,1.0,0.0,8,1501.0,MC,1900-01-01 23:10:46,1900-01-01 23:12:46,0 days 00:02:00,23,10
14729,1904,2026-03-02,14713,403,0.0,0.0,8,1501.0,MC,1900-01-01 23:17:52,1900-01-01 23:18:45,0 days 00:00:53,23,17
14730,1904,2026-03-02,14714,404,0.0,1.0,7,1501.0,MC,1900-01-01 23:19:37,1900-01-01 23:20:17,0 days 00:00:40,23,19
14731,1904,2026-03-02,14715,94,1.0,0.0,8,1501.0,MC,1900-01-01 23:21:17,1900-01-01 23:21:51,0 days 00:00:34,23,21
14732,1904,2026-03-02,14716,95,0.0,0.0,8,1501.0,MC,1900-01-01 23:22:26,1900-01-01 23:22:53,0 days 00:00:27,23,22


In [56]:
# add inbound and outbound labels
# inbound: Carmack stops
# outbound: UH, Doan
df['BOARDING_DIRECTIONS'] = df['STOP_ID'].apply(
    lambda x: 
    'IB' if x in [403, 404, 94, 95] 
    else 'OB' if x in [37, 401] # OB indicates the hospital stops
    else np.nan
)

In [57]:
df[(df['RUN_ID'] == 1501) & (df['DATE'] == '2026-03-02') & (df['BOARDING_DIRECTIONS'] == 'OB')].sort_values(by = 'ARRIVAL').head(20)

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,BOARDING_DIRECTIONS
19486,2104,2026-03-02,19465,401,1.0,6.0,6,1501.0,MC,1900-01-01 05:02:56,1900-01-01 05:03:30,0 days 00:00:34,5,2,OB
19487,2104,2026-03-02,19466,37,0.0,3.0,3,1501.0,MC,1900-01-01 05:03:40,1900-01-01 05:04:44,0 days 00:01:04,5,3,OB
19490,2104,2026-03-02,19469,401,0.0,11.0,15,1501.0,MC,1900-01-01 05:23:10,1900-01-01 05:23:53,0 days 00:00:43,5,23,OB
19491,2104,2026-03-02,19470,37,0.0,8.0,7,1501.0,MC,1900-01-01 05:24:04,1900-01-01 05:25:16,0 days 00:01:12,5,24,OB
19494,2104,2026-03-02,19473,401,1.0,11.0,17,1501.0,MC,1900-01-01 05:42:55,1900-01-01 05:43:46,0 days 00:00:51,5,42,OB
19495,2104,2026-03-02,19474,37,0.0,12.0,5,1501.0,MC,1900-01-01 05:44:03,1900-01-01 05:45:29,0 days 00:01:26,5,44,OB
19498,2104,2026-03-02,19477,37,1.0,0.0,7,1501.0,MC,1900-01-01 06:05:15,1900-01-01 06:07:23,0 days 00:02:08,6,5,OB
19501,2104,2026-03-02,19480,401,0.0,5.0,13,1501.0,MC,1900-01-01 06:26:04,1900-01-01 06:26:39,0 days 00:00:35,6,26,OB
19502,2104,2026-03-02,19481,37,1.0,7.0,7,1501.0,MC,1900-01-01 06:26:53,1900-01-01 06:28:53,0 days 00:02:00,6,26,OB
19505,2104,2026-03-02,19484,401,0.0,0.0,6,1501.0,MC,1900-01-01 06:49:00,1900-01-01 06:49:13,0 days 00:00:13,6,49,OB


In [ ]:
# create a separate time variable to reduce chance of crossing midnight issues
df['TIME'] = pd.to_datetime(df['DATE'] + ' ' + df['ARRIVAL'])

# loop through each run to identify adjacent UH and Doan stops and combine them
pairing_stops = df.sort_values(by = ['RUN_ID', 'TIME']).copy() # sort by run, date, and arrival time to ensure correct sequence

# subset to only OB stops
pairing_stops = pairing_stops[pairing_stops['BOARDING_DIRECTIONS'] == 'OB']
# 12593 rows of OB stops -> should condense to about 6296 rows after combining

# pairing_stops.head(20)
print(len(pairing_stops))

pairing_stops rows: 36110


In [59]:
# walk rows in order and merge when stop_id is 37 or 401
# combine when stop order is 401 -> 37, arrival time is within 5 minutes, and same run_id and date
for i in range(len(pairing_stops) - 1):
    current_stop = pairing_stops.iloc[i] # current stop
    next_stop = pairing_stops.iloc[i + 1] # next stop by time

    if current_stop['STOP_ID'] == 401: # UH stop
        
        if next_stop['STOP_ID'] != 37: # Doan stop
            print(f"UH stop at index {i} not followed by Doan stop. Current stop time: {current_stop['TIME']}, Next stop time: {next_stop['TIME']}")

# 21 instances where there is a UH stop not followed by Doan stop.

UH stop at index 731 not followed by Doan stop. Current stop time: 2026-03-10 23:48:29-01:00, Next stop time: 2026-03-11 05:03:09-01:00
UH stop at index 1653 not followed by Doan stop. Current stop time: 2026-03-23 23:49:16-01:00, Next stop time: 2026-03-24 05:02:52-01:00
UH stop at index 3656 not followed by Doan stop. Current stop time: 2026-03-27 06:22:19-01:00, Next stop time: 2026-03-27 06:45:11-01:00
UH stop at index 4303 not followed by Doan stop. Current stop time: 2026-03-10 23:58:57-01:00, Next stop time: 2026-03-11 00:00:21-01:00
UH stop at index 4464 not followed by Doan stop. Current stop time: 2026-03-12 23:59:27-01:00, Next stop time: 2026-03-13 00:00:00-01:00
UH stop at index 5332 not followed by Doan stop. Current stop time: 2026-03-30 07:52:34-01:00, Next stop time: 2026-03-30 08:11:38-01:00
UH stop at index 6567 not followed by Doan stop. Current stop time: 2026-03-16 19:39:17-01:00, Next stop time: 2026-03-16 19:42:51-01:00
UH stop at index 6675 not followed by Doan

In [60]:
# walk rows in order and merge when stop_id is 37 or 401
# combine when stop order is 401 -> 37, arrival time is within 5 minutes, and same run_id and date
for i in range(1, len(pairing_stops)):
    current_stop = pairing_stops.iloc[i] # current stop
    last_stop = pairing_stops.iloc[i - 1] # last stop by time

    if current_stop['STOP_ID'] == 37: 
        
        if last_stop['STOP_ID'] != 401: 
            sum += 1
            print(f"Doan stop at index {i} not preceded by UH stop. Current stop time: {current_stop['TIME']}, Last stop time: {last_stop['TIME']}")

# 154 instances where there is a Doan stop not preceded by UH stop.

Doan stop at index 6 not preceded by UH stop. Current stop time: 2026-03-02 06:05:15-01:00, Last stop time: 2026-03-02 05:44:03-01:00
Doan stop at index 475 not preceded by UH stop. Current stop time: 2026-03-06 15:52:18-01:00, Last stop time: 2026-03-06 15:30:53-01:00
Doan stop at index 536 not preceded by UH stop. Current stop time: 2026-03-09 07:25:27-01:00, Last stop time: 2026-03-09 07:04:26-01:00
Doan stop at index 575 not preceded by UH stop. Current stop time: 2026-03-09 14:46:46-01:00, Last stop time: 2026-03-09 14:26:03-01:00
Doan stop at index 588 not preceded by UH stop. Current stop time: 2026-03-09 17:22:18-01:00, Last stop time: 2026-03-09 17:00:57-01:00
Doan stop at index 647 not preceded by UH stop. Current stop time: 2026-03-10 08:36:47-01:00, Last stop time: 2026-03-10 08:12:29-01:00
Doan stop at index 648 not preceded by UH stop. Current stop time: 2026-03-10 08:55:39-01:00, Last stop time: 2026-03-10 08:36:47-01:00
Doan stop at index 920 not preceded by UH stop. Cu

In [73]:
# Build a dataframe with the same columns as pairing_stops, except valid 401->37 pairs become one 999 row.
work = pairing_stops.sort_values(by=['RUN_ID', 'TIME']).reset_index(drop=True).copy()
combined_rows = []
i = 0

while i < len(work):
    current = work.iloc[i]

    can_check_next = i + 1 < len(work)
    if not can_check_next:
        combined_rows.append(current.to_dict())
        break

    nxt = work.iloc[i + 1]

    same_run = current['RUN_ID'] == nxt['RUN_ID']
    same_date = current['DATE'] == nxt['DATE']
    same_bus = True if 'BUS_ID' not in work.columns else current['BUS_ID'] == nxt['BUS_ID']
    in_order = (current['STOP_ID'] == 401) and (nxt['STOP_ID'] == 37)
    gap = nxt['TIME'] - current['TIME']
    within_gap = pd.Timedelta(0) <= gap <= pd.Timedelta(minutes=max_gap_minutes)

    if in_order and same_run and same_date and same_bus and within_gap:
        merged = current.copy()
        merged['STOP_ID'] = combined_stop_id

        if 'BOARDINGS' in work.columns:
            merged['BOARDINGS'] = current['BOARDINGS'] + nxt['BOARDINGS']
        if 'ALIGHTINGS' in work.columns:
            merged['ALIGHTINGS'] = current['ALIGHTINGS'] + nxt['ALIGHTINGS']
        if 'LOAD' in work.columns:
            merged['LOAD'] = max(current['LOAD'], nxt['LOAD'])
        if 'DEPARTURE' in work.columns:
            merged['DEPARTURE'] = nxt['DEPARTURE']
        if 'DWELL' in work.columns and 'ARRIVAL' in work.columns and 'DEPARTURE' in work.columns:
            arr_ts = pd.to_datetime(str(merged['ARRIVAL']), errors='coerce')
            dep_ts = pd.to_datetime(str(merged['DEPARTURE']), errors='coerce')
            merged['DWELL'] = dep_ts - arr_ts if pd.notna(arr_ts) and pd.notna(dep_ts) else pd.NaT
        if 'HOUR' in work.columns and 'ARRIVAL' in work.columns:
            arr_ts = pd.to_datetime(str(merged['ARRIVAL']), errors='coerce')
            merged['HOUR'] = int(arr_ts.hour) if pd.notna(arr_ts) else merged.get('HOUR', np.nan)
        if 'MINUTE' in work.columns and 'ARRIVAL' in work.columns:
            arr_ts = pd.to_datetime(str(merged['ARRIVAL']), errors='coerce')
            merged['MINUTE'] = int(arr_ts.minute) if pd.notna(arr_ts) else merged.get('MINUTE', np.nan)

        combined_rows.append(merged.to_dict())
        i += 2
    else:
        combined_rows.append(current.to_dict())
        i += 1

pairing_stops_combined = pd.DataFrame(combined_rows, columns=pairing_stops.columns)

print('Original rows:', len(pairing_stops))
print('Combined rows:', len(pairing_stops_combined))
print('Remaining 401/37 rows:', pairing_stops_combined['STOP_ID'].isin([401, 37]).sum())
print('New 999 rows:', (pairing_stops_combined['STOP_ID'] == 999).sum())

Original rows: 36110
Combined rows: 29958
Remaining 401/37 rows: 289
New 999 rows: 6152


In [74]:
pairing_stops_combined['STOP_ID'].value_counts()

STOP_ID
404    6417
403    6411
999    6152
95     5360
94     5329
37      211
401      78
Name: count, dtype: int64

In [75]:
# maybe try converting 37 and 401 to 999 to account for the missing pairs, compare to existing results when done
pairing_stops_combined['STOP_ID'] = pairing_stops_combined['STOP_ID'].replace({37: 999, 401: 999})

In [76]:
pairing_stops_combined

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,TIME
0,2104,2026-03-02,19463,403,7.0,1.0,8,1501.0,MC,1900-01-01 04:53:40,1900-01-01 04:55:41,0 days 00:02:01,4,53,2026-03-02 04:53:40
1,2104,2026-03-02,19464,404,3.0,0.0,11,1501.0,MC,1900-01-01 04:56:24,1900-01-01 04:57:57,0 days 00:01:33,4,56,2026-03-02 04:56:24
2,2104,2026-03-02,19465,999,1.0,9.0,6,1501.0,MC,1900-01-01 05:02:56,1900-01-01 05:04:44,0 days 00:01:48,5,2,2026-03-02 05:02:56
3,2104,2026-03-02,19467,403,9.0,1.0,14,1501.0,MC,1900-01-01 05:10:17,1900-01-01 05:14:37,0 days 00:04:20,5,10,2026-03-02 05:10:17
4,2104,2026-03-02,19468,404,10.0,0.0,26,1501.0,MC,1900-01-01 05:15:09,1900-01-01 05:16:36,0 days 00:01:27,5,15,2026-03-02 05:15:09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29953,1902,2026-03-30,14517,403,0.0,1.0,1,1514.0,MC,1900-01-01 07:11:07,1900-01-01 07:11:42,0 days 00:00:35,7,11,2026-03-30 07:11:07
29954,1902,2026-03-30,14518,404,8.0,0.0,9,1514.0,MC,1900-01-01 07:13:01,1900-01-01 07:14:08,0 days 00:01:07,7,13,2026-03-30 07:13:01
29955,1902,2026-03-30,14519,94,2.0,0.0,11,1514.0,MC,1900-01-01 07:15:04,1900-01-01 07:15:37,0 days 00:00:33,7,15,2026-03-30 07:15:04
29956,1902,2026-03-30,14520,95,1.0,0.0,12,1514.0,MC,1900-01-01 07:16:06,1900-01-01 07:16:42,0 days 00:00:36,7,16,2026-03-30 07:16:06
